# FILTRO CON ENTRENAMIENTO

In [2]:
import random
import time
from collections import deque
from pathlib import Path
from datetime import datetime
import pandas as pd

import cv2
import mediapipe as mp
import numpy as np

from utilidades.extraer_caracteristicas import crear_landmarker, resultado_a_arrays, indice_cara_mayor, caja_cara
from utilidades.modelo_emociones import PredictorEmociones

In [ ]:
# Configuración y carga del modelo entrenado
RUTA_MODELO_MP = Path("modelos_externos") / "face_landmarker.task"
RUTA_RED = Path("entrenamiento") / "ambos" / "modelo.pt"
RUTA_RESULTADOS_WEBCAM = Path("pruebas_webcam") / "resultados_webcam.csv"
CAMARA = 0
SEG_CALIBRACION = 3.0    # segundos con cara neutra al empezar
VENTANA_SUAVIZADO = 10     # fotogramas promediados (~0,3 s a 30 FPS)

predictor = PredictorEmociones(RUTA_RED)
print("Tipo de características:", predictor.tipo)

Tipo de características: ambos


In [9]:
# Funciones de dibujo
FUENTE = cv2.FONT_HERSHEY_SIMPLEX

COLORES = {
    "alegria":  (0, 215, 255),
    "asco":     (0, 160, 0),
    "enfado":   (0, 0, 230),
    "miedo":    (180, 0, 140),
    "neutral":  (160, 160, 160),
    "sorpresa": (0, 140, 255),
    "tristeza": (230, 120, 0),
}

# Posiciones verticales de la interfaz
Y_BANDA = 45    # banda superio
Y_ORDEN = 95    # texto de la orden
Y_TIEMPO = 112   # barra de tiempo restante
Y_MANTENER = 132   # barra de cuánto se mantiene la emoción
Y_BARRAS = 160   # barras de probabilidad


# Texto simple
def texto(img, msg, pos, escala=0.8, color=(255, 255, 255), grosor=2):
    cv2.putText(img, msg, pos, FUENTE, escala, color, grosor, cv2.LINE_AA)


# Centra el texto y si no cabe en la ventana se reduce
def centrado(img, msg, y, escala, color, grosor=2, margen=20):
    ancho = img.shape[1]
    (w, _), _ = cv2.getTextSize(msg, FUENTE, escala, grosor)
    if w > ancho - 2 * margen:
        escala *= (ancho - 2 * margen) / w
        (w, _), _ = cv2.getTextSize(msg, FUENTE, escala, grosor)
    texto(img, msg, ((ancho - w) // 2, y), escala, color, grosor)

# Oscurece una franja horizontal
def oscurecer(img, y1, y2, alfa=0.55):
    img[y1:y2] = (img[y1:y2] * (1 - alfa)).astype(np.uint8)


# Barra de progreso
def barra(img, x, y, w, h, frac, color):
    cv2.rectangle(img, (x, y), (x + w, y + h), (50, 50, 50), -1)
    cv2.rectangle(img, (x, y), (x + int(w * max(0.0, min(1.0, frac))), y + h), color, -1)
    cv2.rectangle(img, (x, y), (x + w, y + h), (255, 255, 255), 1)


# Enseña la probabilidad de cada emoción en forma de barras
def dibujar_barras(img, emociones, probs, x0=10, y0=Y_BARRAS, ancho_max=220, alto=22):
    for i, (emo, p) in enumerate(zip(emociones, probs)):
        y = y0 + i * (alto + 6)
        cv2.rectangle(img, (x0, y), (x0 + ancho_max, y + alto), (50, 50, 50), -1)
        cv2.rectangle(img, (x0, y), (x0 + int(p * ancho_max), y + alto), COLORES.get(emo, (200, 200, 200)), -1)
        cv2.putText(img, f"{emo} {p * 100:3.0f}%", (x0 + 5, y + alto - 6), FUENTE, 0.55,
                    (255, 255, 255), 1, cv2.LINE_AA)


# Banda superior: vidas a la izquierda, ronda y racha en el centro, puntos a la derecha
def dibujar_hud(img, juego):
    ancho = img.shape[1]
    oscurecer(img, 0, Y_BANDA)
    for i in range(juego.vidas_iniciales):
        color = (0, 0, 230) if i < juego.vidas else (80, 80, 80)
        cv2.circle(img, (25 + i * 30, Y_BANDA // 2), 10, color, -1)
        cv2.circle(img, (25 + i * 30, Y_BANDA // 2), 10, (255, 255, 255), 1)
    centrado(img, f"Ronda {juego.ronda + 1}   Racha {juego.racha}", 30, 0.6, (255, 255, 255), 2)
    msg = f"Puntos: {juego.puntos}"
    (w, _), _ = cv2.getTextSize(msg, FUENTE, 0.75, 2)
    texto(img, msg, (ancho - w - 15, 31), 0.75)


In [5]:
# Lógica del Simón dice: órdenes, tiempos, vidas y puntuación (sin cámara ni dibujo).
# Estados: "volver_neutral" (espera cara neutra), "orden", "resultado" y "fin".
class JuegoSimon:
    def __init__(self, emociones_modelo, emociones_juego, umbrales, umbrales_trampa,
                 prob_trampa=0.3, vidas=3,
                 tiempo_inicial=6.0, tiempo_minimo=3.0, reduccion_por_ronda=0.3,
                 seg_mantener=0.5, seg_trampa=0.4,
                 umbral_neutral=0.5, seg_neutral=0.5, max_espera_neutral=4.0,
                 seg_resultado=1.5):
        self.emociones_modelo = list(emociones_modelo)
        self.emociones_juego = list(emociones_juego)
        self.umbrales = umbrales     # para acertar una orden con "Simón dice"
        self.umbrales_trampa = umbrales_trampa    # a partir de cuánto se pica en una trampa
        self.prob_trampa = prob_trampa
        self.vidas_iniciales = vidas
        self.tiempo_inicial = tiempo_inicial
        self.tiempo_minimo = tiempo_minimo
        self.reduccion_por_ronda = reduccion_por_ronda
        self.seg_mantener = seg_mantener     # segundos que hay que sostener la emoción
        self.seg_trampa = seg_trampa         # segundos sosteniéndola para picar
        self.umbral_neutral = umbral_neutral
        self.seg_neutral = seg_neutral
        self.max_espera_neutral = max_espera_neutral
        self.seg_resultado = seg_resultado
        self.reiniciar()

    def reiniciar(self):
        self.vidas = self.vidas_iniciales
        self.puntos = 0
        self.ronda = 0
        self.racha = 0
        self.mejor_racha = 0
        self.emocion = None
        self.es_simon = None
        self.mensaje, self.acierto = "", None
        self._ir_a_neutral()

    def _ir_a_neutral(self):
        self.estado = "volver_neutral"
        self.t_estado = 0.0      # tiempo en el estado actual
        self.t_sostenido = 0.0   # tiempo seguido por encima del umbral

    # El tiempo de cada orden baja con las rondas hasta un mínimo
    @property
    def tiempo_limite(self):
        return max(self.tiempo_minimo, self.tiempo_inicial - self.reduccion_por_ronda * self.ronda)

    @property
    def tiempo_restante(self):
        return max(0.0, self.tiempo_limite - self.t_estado) if self.estado == "orden" else 0.0

    # De 0 a 1: cuánto lleva el jugador sosteniendo la emoción pedida
    @property
    def progreso_mantener(self):
        objetivo = self.seg_mantener if self.es_simon else self.seg_trampa
        return min(1.0, self.t_sostenido / objetivo) if self.estado == "orden" else 0.0

    def texto_orden(self):
        base = f"cara de {self.emocion.upper()}!"
        return f"Simon dice: {base}" if self.es_simon else "C" + base[1:]

    def _nueva_orden(self):
        opciones = [e for e in self.emociones_juego if e != self.emocion]
        self.emocion = random.choice(opciones)
        self.es_simon = random.random() >= self.prob_trampa
        self.estado = "orden"
        self.t_estado = 0.0
        self.t_sostenido = 0.0

    def _terminar_ronda(self, acierto, mensaje, puntos=0):
        self.acierto, self.mensaje = acierto, mensaje
        self.ronda += 1
        if acierto:
            self.racha += 1
            self.mejor_racha = max(self.mejor_racha, self.racha)
            self.puntos += puntos
        else:
            self.racha = 0
            self.vidas -= 1
        self.estado = "resultado"
        self.t_estado = 0.0

    # Avanza el juego un fotograma. probs = None si no se ve la cara (el juego se pausa).
    def actualizar(self, probs, dt):
        if self.estado == "fin":
            return

        if self.estado == "resultado":
            self.t_estado += dt
            if self.t_estado >= self.seg_resultado:
                if self.vidas <= 0:
                    self.estado = "fin"
                else:
                    self._ir_a_neutral()
            return

        if probs is None:
            self.t_sostenido = 0.0
            return

        # Antes de cada orden: cara neutra sostenida (o espera máxima)
        if self.estado == "volver_neutral":
            self.t_estado += dt
            p_neutral = probs[self.emociones_modelo.index("neutral")]
            self.t_sostenido = self.t_sostenido + dt if p_neutral >= self.umbral_neutral else 0.0
            if self.t_sostenido >= self.seg_neutral or self.t_estado >= self.max_espera_neutral:
                self._nueva_orden()
            return

        # Orden activa
        self.t_estado += dt
        p = probs[self.emociones_modelo.index(self.emocion)]

        if self.es_simon:
            # Acierta si mantiene la emoción pedida antes de que se acabe el tiempo
            self.t_sostenido = self.t_sostenido + dt if p >= self.umbrales[self.emocion] else 0.0
            if self.t_sostenido >= self.seg_mantener:
                rapidez = self.tiempo_restante / self.tiempo_limite
                puntos = 100 + int(100 * rapidez) + 10 * self.racha
                self._terminar_ronda(True, f"Bien! +{puntos}", puntos)
            elif self.t_estado >= self.tiempo_limite:
                self._terminar_ronda(False, "Demasiado lento!")
        else:
            # Falla si pone la emoción; acierta si aguanta hasta el final
            self.t_sostenido = self.t_sostenido + dt if p >= self.umbrales_trampa[self.emocion] else 0.0
            if self.t_sostenido >= self.seg_trampa:
                self._terminar_ronda(False, "Simon no lo ha dicho!")
            elif self.t_estado >= self.tiempo_limite:
                puntos = 50 + 10 * self.racha
                self._terminar_ronda(True, f"No has picado! +{puntos}", puntos)

In [10]:
# Prueba de webcam (con todas las emociones)
PERSONA_PRUEBA = "persona1"
CONDICION = "luz_tenue"
ORDEN_PRUEBA = ["neutral", "alegria", "tristeza", "enfado", "asco", "miedo", "sorpresa"]
SEG_GRABACION = 3.0   # segundos que se mide cada emoción
RUTA_RESULTADOS_WEBCAM.parent.mkdir(exist_ok=True)

cap = cv2.VideoCapture(CAMARA)
if not cap.isOpened():
    raise RuntimeError("No se pudo abrir la cámara. Prueba con CAMARA = 1.")

probs_recientes = deque(maxlen=VENTANA_SUAVIZADO)
predictor.olvidar_calibracion()
estado = "calibrando"
inicio_calib, muestras_calib = None, []
indice, inicio_grab = 0, None
grabadas = {} 
ultimo_ts = 0
resumen = None

with crear_landmarker(RUTA_MODELO_MP, modo="video") as lmk:
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        alto, ancho = frame.shape[:2]
        ts = max(int(time.perf_counter() * 1000), ultimo_ts + 1)
        ultimo_ts = ts
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        res = lmk.detect_for_video(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb), ts)
        cara = resultado_a_arrays(res, ancho, alto)
        vista = cv2.flip(frame, 1)
        objetivo = ORDEN_PRUEBA[indice] if indice < len(ORDEN_PRUEBA) else None

        # Calibración de la cara neutra
        if estado == "calibrando":
            if cara is None:
                inicio_calib = None
                centrado(vista, "Mira a la camara", alto // 2, 1.0, (0, 0, 255))
            else:
                if inicio_calib is None:
                    inicio_calib, muestras_calib = time.perf_counter(), []
                muestras_calib.append(cara)
                restante = SEG_CALIBRACION - (time.perf_counter() - inicio_calib)
                texto(vista, f"Calibrando: pon cara neutra {max(restante, 0):.1f}s", (20, 40), 0.8, (0, 255, 255))
                if restante <= 0:
                    predictor.calibrar(muestras_calib)
                    estado = "esperando"

        # Medición de cada emoción
        elif estado in ("esperando", "grabando"):
            if cara is None:
                probs_recientes.clear()
                centrado(vista, "Mira a la camara", alto // 2, 1.0, (0, 0, 255))
            else:
                probs_recientes.append(predictor.predecir(cara))
                media = np.mean(probs_recientes, axis=0)
                dibujar_barras(vista, predictor.emociones, media, y0=60)
                if estado == "grabando":
                    grabadas[objetivo].append(media)

            texto(vista, f"[{indice + 1}/{len(ORDEN_PRUEBA)}] Pon cara de: {objetivo.upper()}",
                  (20, 40), 0.9, COLORES[objetivo])
            if estado == "esperando":
                texto(vista, "Pulsa ESPACIO cuando estes listo/a", (20, alto - 40), 0.7, (0, 255, 255))
            else:
                restante = SEG_GRABACION - (time.perf_counter() - inicio_grab)
                texto(vista, f"Midiendo... {max(restante, 0):.1f}s", (20, alto - 40), 0.8, (0, 0, 255))
                if restante <= 0:
                    indice += 1
                    estado = "esperando" if indice < len(ORDEN_PRUEBA) else "fin"

        # Al terminar se calcula el acierto de cada emoción y se añade al CSV
        if estado == "fin":
            if resumen is None:
                filas = []
                for emo in ORDEN_PRUEBA:
                    p = np.array(grabadas[emo])
                    i_emo = predictor.emociones.index(emo)
                    ganadoras = p.argmax(1)
                    filas.append({"pedida": emo, "fotogramas": len(p),
                                  "acierto_pct": round(100 * (ganadoras == i_emo).mean(), 1),
                                  "prob_media_pedida": round(float(p[:, i_emo].mean()), 3),
                                  "detectada_mas_veces": predictor.emociones[np.bincount(ganadoras).argmax()]})
                resumen = pd.DataFrame(filas)
                resumen.insert(0, "condicion", CONDICION)
                resumen.insert(0, "persona", PERSONA_PRUEBA)
                resumen.insert(0, "fecha", datetime.now().strftime("%Y-%m-%d %H:%M"))
                resumen.to_csv(RUTA_RESULTADOS_WEBCAM, mode="a", index=False,
                               header=not RUTA_RESULTADOS_WEBCAM.exists())
            vista[:] = (vista * 0.35).astype(np.uint8)
            for i, fila in enumerate(resumen.itertuples()):
                color = (0, 220, 0) if fila.acierto_pct >= 50 else (0, 0, 255)
                texto(vista, f"{fila.pedida:9s} {fila.acierto_pct:5.1f}%  ->  {fila.detectada_mas_veces}",
                      (20, 60 + i * 32), 0.6, color)

        texto(vista, "ESPACIO: medir  R: repetir anterior  ESC: salir", (10, alto - 12), 0.5)
        cv2.imshow("Prueba guiada", vista)

        tecla = cv2.waitKey(1) & 0xFF
        if tecla == 27:
            break
        if tecla == 32 and estado == "esperando" and cara is not None:   # ESPACIO
            probs_recientes.clear()
            grabadas[objetivo] = []
            inicio_grab, estado = time.perf_counter(), "grabando"
        if tecla in (ord("r"), ord("R")) and estado == "esperando" and indice > 0:
            indice -= 1

cap.release()
cv2.destroyAllWindows()

In [7]:
# Resumen de todas las pruebas con webcam
res_web = pd.read_csv(RUTA_RESULTADOS_WEBCAM)
res_web["gafas"] = res_web["condicion"].str.contains("gafas")

print(f"Pruebas: {res_web.groupby(['fecha', 'persona', 'condicion']).ngroups} · "
      f"personas: {res_web['persona'].nunique()}\n")
print("Acierto medio por emoción (%):")
print(res_web.groupby("pedida")["acierto_pct"].mean().round(1).sort_values(ascending=False).to_string())

# Con y sin gafas por persona, para comparar solo pruebas de la misma persona
print("\nAcierto medio por persona, con y sin gafas (%):")
print(res_web.pivot_table(index="persona", columns="gafas", values="acierto_pct", aggfunc="mean").round(1))

Pruebas: 21 · personas: 6

Acierto medio por emoción (%):
pedida
alegria     98.1
neutral     90.5
enfado      88.3
tristeza    88.3
sorpresa    68.9
miedo       57.3
asco        35.5

Acierto medio por persona, con y sin gafas (%):
gafas     False  True 
persona               
persona1   93.2   93.5
persona2   81.5   84.0
persona3   50.6    NaN
persona4   77.9   73.5
persona5   73.4    NaN
persona6   51.8   46.5


In [11]:
# Simón dice con la webcam
EMOCIONES_JUEGO = ["alegria", "tristeza", "enfado", "sorpresa", "miedo", "asco"]

# Probabilidad necesaria para acertar una orden con "Simón dice"
UMBRALES = {"alegria": 0.6, "tristeza": 0.6, "enfado": 0.6, "sorpresa": 0.6,
            "miedo": 0.45, "asco": 0.45}
# Probabilidad a partir de la cual se pica en una trampa
UMBRALES_TRAMPA = {"alegria": 0.7, "tristeza": 0.7, "enfado": 0.7, "sorpresa": 0.7,
                   "miedo": 0.6, "asco": 0.6}

MOSTRAR_BARRAS = False

juego = JuegoSimon(predictor.emociones, EMOCIONES_JUEGO, UMBRALES, UMBRALES_TRAMPA)

cap = cv2.VideoCapture(CAMARA)
if not cap.isOpened():
    raise RuntimeError("No se pudo abrir la cámara. Prueba con CAMARA = 1.")

probs_recientes = deque(maxlen=VENTANA_SUAVIZADO)
predictor.olvidar_calibracion()
calibrando = True
inicio_calib, muestras_calib = None, []
ultimo_ts, t_prev = 0, time.perf_counter()

with crear_landmarker(RUTA_MODELO_MP, modo="video", num_caras=3) as lmk:
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        alto, ancho = frame.shape[:2]
        ahora = time.perf_counter()
        dt, t_prev = min(ahora - t_prev, 0.2), ahora   # acotado por si hay un parón

        # Mediapipe exige marcas de tiempo crecientes
        ts = max(int(ahora * 1000), ultimo_ts + 1)
        ultimo_ts = ts
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        res = lmk.detect_for_video(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb), ts)

        # El jugador es la cara más cercana; Mediapipe trabaja sin espejo, como en KDEF
        idx_cara = indice_cara_mayor(res)
        cara = None if idx_cara is None else resultado_a_arrays(res, ancho, alto, idx_cara)
        vista = cv2.flip(frame, 1)   # espejo solo para mostrar

        n_caras = len(res.face_landmarks)
        if n_caras > 1:
            # Recuadro en el jugador
            x1, y1, x2, y2 = caja_cara(res, idx_cara, ancho, alto)
            cv2.rectangle(vista, (ancho - x2, y1), (ancho - x1, y2), (0, 255, 255), 2)
            texto(vista, f"{n_caras} caras: juega la mas cercana", (10, alto - 45), 0.55, (0, 255, 255))

        # Calibración de la cara neutra
        if calibrando:
            oscurecer(vista, 0, Y_BANDA + 30)
            centrado(vista, "SIMON DICE", 50, 1.4, (0, 255, 255), 3)
            if cara is None:
                inicio_calib = None   # si se pierde la cara, se empieza de nuevo
                centrado(vista, "Mira a la camara", alto // 2, 1.0, (0, 0, 255))
            else:
                if inicio_calib is None:
                    inicio_calib, muestras_calib = ahora, []
                muestras_calib.append(cara)
                restante = SEG_CALIBRACION - (ahora - inicio_calib)
                centrado(vista, "Pon cara neutra y no te muevas", alto // 2, 0.9, (255, 255, 255))
                centrado(vista, f"{max(restante, 0):.1f}", alto // 2 + 50, 1.2, (0, 255, 255))
                if restante <= 0:
                    predictor.calibrar(muestras_calib)
                    calibrando = False
                    probs_recientes.clear()
                    juego.reiniciar()
                    t_prev = time.perf_counter()

        # Partida
        else:
            if cara is None:
                probs_recientes.clear()
                media = None
            else:
                probs_recientes.append(predictor.predecir(cara))
                media = np.mean(probs_recientes, axis=0)   # suavizado temporal

            juego.actualizar(media, dt)

            if juego.estado != "fin":
                dibujar_hud(vista, juego)
                if media is None:
                    centrado(vista, "Mira a la camara (pausa)", alto // 2 + 120, 0.8, (0, 0, 255))
                elif MOSTRAR_BARRAS:
                    dibujar_barras(vista, predictor.emociones, media)

            if juego.estado == "volver_neutral":
                centrado(vista, "Vuelve a cara neutra...", Y_ORDEN, 1.0, (200, 200, 200))
                if juego.t_estado > 2.5:
                    centrado(vista, "(si no la detecta, pulsa C para recalibrar)", Y_ORDEN + 30,
                             0.55, (200, 200, 200), 2)

            elif juego.estado == "orden":
                # Mismo color con o sin "Simón dice", para no dar pistas
                color = COLORES[juego.emocion]
                centrado(vista, juego.texto_orden(), Y_ORDEN, 1.1, color, 3)
                barra(vista, ancho // 2 - 200, Y_TIEMPO, 400, 12,
                      juego.tiempo_restante / juego.tiempo_limite, (0, 200, 255))
                barra(vista, ancho // 2 - 200, Y_MANTENER, 400, 8, juego.progreso_mantener, color)

            elif juego.estado == "resultado":
                color = (0, 220, 0) if juego.acierto else (0, 0, 255)
                cv2.rectangle(vista, (0, 0), (ancho - 1, alto - 1), color, 12)
                centrado(vista, juego.mensaje, alto // 2, 1.4, color, 3)

            elif juego.estado == "fin":
                vista[:] = (vista * 0.35).astype(np.uint8)
                centrado(vista, "FIN DE LA PARTIDA", alto // 2 - 60, 1.4, (0, 0, 255), 3)
                centrado(vista, f"Puntos: {juego.puntos}", alto // 2, 1.1, (255, 255, 255))
                centrado(vista, f"Rondas: {juego.ronda}   Mejor racha: {juego.mejor_racha}",
                         alto // 2 + 45, 0.8, (255, 255, 255))
                centrado(vista, "R: jugar otra vez   ESC: salir", alto // 2 + 100, 0.7, (0, 255, 255))

        oscurecer(vista, alto - 28, alto, 0.5)
        texto(vista, "ESC: salir  C: recalibrar  R: reiniciar  B: barras", (10, alto - 10), 0.5)
        cv2.imshow("Simon dice", vista)

        # Teclas
        tecla = cv2.waitKey(1) & 0xFF
        if tecla == 27:
            break
        if tecla in (ord("c"), ord("C")):
            predictor.olvidar_calibracion()
            calibrando, inicio_calib = True, None
            probs_recientes.clear()
        if tecla in (ord("r"), ord("R")) and not calibrando:
            juego.reiniciar()
            probs_recientes.clear()
        if tecla in (ord("b"), ord("B")):
            MOSTRAR_BARRAS = not MOSTRAR_BARRAS

cap.release()
cv2.destroyAllWindows()